# Clase 1 — Agente de Viajes Corporativos

En este notebook construimos un agente completo, paso a paso:

1. **Concepto** — qué es la IA Agentiva
2. **Tool simple** — definir e integrar una herramienta con `@tool`
3. **Tools reales** — buscar vuelos (Duffel API) y clima (Open-Meteo)
4. **Agente completo** — combinar tools + system prompt + conversación multi-turno

## Setup

Asegurate de tener las variables de entorno en un archivo `.env` en este directorio:

```
DUFFEL_API_KEY=duffel_sandbox_...
```

La siguiente celda instala las dependencias e importa las utilidades necesarias.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv()
print('Setup OK')
!pip install -r requirements.txt

## ¿Qué es la IA Agentiva?

Antes de escribir código, le preguntamos al propio agente que explique el concepto. Un agente va más allá de la pregunta-respuesta simple: puede **planificar**, **llamar herramientas** y **completar tareas en múltiples pasos** de forma autónoma.

In [ ]:
from strands import Agent

simple_agent = Agent()  
result = simple_agent('What is Agentic AI? Explain in 2 sentences.')

## Creando una Tool simple con `@tool`

El decorador `@tool` convierte cualquier función Python en una herramienta que el agente puede usar. Strands lee automáticamente:

- El **nombre** de la función → `tool_name`
- El **docstring** → descripción que el modelo lee para saber cuándo usarla
- Los **type hints** → schema JSON de parámetros y tipos

En este ejemplo la tool devuelve siempre 25°C. En producción, aquí iría la llamada a una API real.

In [ ]:
from strands import tool

@tool
def get_temperature(city: str) -> str:
    """Get the current temperature of a city.
    Args: city: City name, e.g., "Buenos Aires" 
    """
    # Aquí iría la lógica para obtener la temperatura real, por ejemplo, llamando a una API de clima.
    return f"The current temperature in {city} is 25°C."

print('Tool name :', get_temperature.tool_name)

## Integrando la tool al agente

Pasamos la lista de tools al constructor de `Agent`. El modelo ahora sabe que puede llamar `get_temperature` y lo hace automáticamente cuando el usuario pregunta por el clima. Notá el output `Tool #1: get_temperature` que Strands imprime al ejecutar la tool.

In [ ]:
agente = Agent(tools=[get_temperature])

agente("Cual es la temperatura actual en Lima, Peru?")

## Tools de producción

En lugar de funciones dummy, importamos tres tools reales del módulo `tools/`:

| Tool | Descripción | API |
|------|-------------|-----|
| `search_flights` | Busca vuelos one-way ordenados por precio | Duffel Sandbox |
| `get_weather` | Pronóstico del tiempo por ciudad y fecha | Open-Meteo (gratis) |
| `save_search_log` | Persiste la búsqueda en un archivo JSONL | Disco local |

Inspeccionamos las claves del `tool_spec` que Strands generó automáticamente para cada una.

In [ ]:
from tools import search_flights, get_weather, save_search_log

for t in [search_flights, get_weather, save_search_log]:
    print(f'{t.tool_name}: {list(t.tool_spec.keys())}')

## Armando el Agente de Viajes

El `system_prompt` define el rol y las reglas del agente. Le indicamos:

- Qué tipo de asistente es (viajes corporativos)
- Qué tools tiene y cuándo usarlas
- Una regla de negocio: **siempre llamar `save_search_log` al final de cada búsqueda**

Un buen system prompt reemplaza instrucciones repetidas en cada mensaje del usuario.

In [ ]:
SYSTEM_PROMPT = """ You are a corporate travel assistant. You help employees find flights and check weather at destination.
Always call save_search_log at the end of each serarch.
Quote price with currency. Never invent flight """


agente2 = Agent(tools=[search_flights, get_weather, save_search_log], 
                system_prompt=SYSTEM_PROMPT)

## Conversación multi-turno: el agente pide información faltante

Le damos una instrucción incompleta (falta el origen). El agente detecta que le falta información esencial y **hace una pregunta de clarificación** en lugar de inventar un aeropuerto de origen.

Este comportamiento emerge naturalmente del modelo, sin lógica adicional de nuestra parte.

In [ ]:
response = agente2("Quiero un vuelo en economica a San Francisco el 25 de Mayo de 2026")

## El agente encadena múltiples tools automáticamente

Al dar el origen, el agente ejecuta la siguiente secuencia en un solo turno:

1. Llama a `search_flights(origin='EZE', destination='SFO', departure_date='2026-05-25')`
2. Recibe los 5 vuelos más baratos
3. Llama a `save_search_log(...)` — regla del system prompt
4. Presenta los resultados formateados al usuario

Todo esto ocurre dentro del **event loop** de Strands, que itera ciclos de razonamiento + acción.

In [ ]:
agente2("Vuelo desde Ezeiza")

## Memoria de contexto: el agente recuerda el destino

Continuamos la conversación con una pregunta vaga ("el destino"). El agente infiere que nos referimos a San Francisco el 25 de mayo — la información del turno anterior todavía está en su contexto — y llama a `get_weather` con esos parámetros.

Esto ilustra el poder de la conversación multi-turno: el estado se mantiene en el historial de mensajes sin necesidad de código adicional.

In [ ]:
agente2("Decime cual va a ser el clima en destino?")